# Class 1 & 2: NLP and Search
## Learning Notebook Part 2 - Advanced Search & Clustering

**Welcome to Part 2!** In this notebook, we'll build on the foundation from Part 1:

**You should have completed Part 1** where you learned:
- Keyword search (simple and multiple keyword search)
- Simple tokenization
- Text preprocessing (cleaning, tokenization, regex)
- Bag of Words (word counts - converting text to numbers)
- Understanding vector representations

**Now in Part 2, you'll learn:**
- 📊 **TF-IDF**: Improved text representation (concept - you'll implement in exercises!)
- 🎯 **Similarity-Based Search**: Finding relevant documents using TF-IDF + cosine similarity
- 📦 **Clustering**: Automatically organizing documents with K-Means

Let's dive in!


## Setup


In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

# For better output display
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"


### Load the Data


In [ ]:
# Load movie descriptions
# If running in Google Colab and data file doesn't exist, download it from GitHub
import os

if not os.path.exists('data/movies.csv'):
    print("Data file not found. Downloading from GitHub...")
    os.makedirs('data', exist_ok=True)
    import urllib.request
    url = 'https://raw.githubusercontent.com/samsung-ai-course/8th-9th-edition/main/Chapter%202%20-%20Natural%20Language%20Processing/Class%201%20%26%202%20-%20NLP%20and%20Search/data/movies.csv'
    urllib.request.urlretrieve(url, 'data/movies.csv')
    print("✓ Data file downloaded successfully!")

df = pd.read_csv('data/movies.csv')
print(f"Loaded {len(df)} movies")
df.head()

## Sparse vs Dense Vectors (Quick Reminder)

Before we dive into TF-IDF, let's quickly visualize sparse vs dense vectors:


In [ ]:
# Example: Sparse vector (TF-IDF will create this)
# Let's use a small example first
sample_docs = df['description'].head(3).tolist()

# Create TF-IDF vectors (sparse!)
vectorizer = TfidfVectorizer(max_features=50, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(sample_docs)

print("Sparse TF-IDF Matrix Shape:", tfidf_matrix.shape)
print(f"Total elements: {tfidf_matrix.shape[0] * tfidf_matrix.shape[1]}")
print(f"Non-zero elements: {tfidf_matrix.nnz}")
print(f"Sparsity: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")
print("\nFirst document vector (first 20 values):")
print(tfidf_matrix[0].toarray()[0][:20])

# Compare to dense vector (random example)
dense_example = np.random.rand(50)
print(f"\n\nDense vector (50 dimensions, all non-zero):")
print(dense_example[:20])


**Key Takeaway**: 
- **Sparse vectors** (BoW, TF-IDF) are interpretable - we know what each dimension means
  - **BoW** (from Part 1): Word counts (e.g., "python" appears 3 times)
  - **TF-IDF** (this part): Word frequencies weighted by importance
- **Dense vectors** (embeddings) are more powerful - they capture meaning and relationships
- For now, we'll use **TF-IDF** (sparse) as it's simple and interpretable. Next class, we'll see dense embeddings!


## TF-IDF: Term Frequency-Inverse Document Frequency

**TF-IDF** is an improvement over simple Bag of Words. It weighs words by:
- **TF (Term Frequency)**: How often a word appears in a document (higher = more important to that document)
- **IDF (Inverse Document Frequency)**: How rare a word is across all documents (rare words are more informative)

### The TF-IDF Formula (Step by Step)

**TF-IDF = TF × IDF**

Let's break this down:

#### 1. Term Frequency (TF)

**TF** measures how frequently a term appears in a document. There are several ways to calculate it:

**Raw Count (Simple):**
```
TF(t, d) = count(t, d)
```
- `t` = term (word)
- `d` = document
- Simply counts how many times term `t` appears in document `d`

**Normalized TF (More Common):**
```
TF(t, d) = count(t, d) / total_words_in_document(d)
```
- Normalizes by document length
- Prevents longer documents from having higher scores just because they have more words

**Log-Scaled TF (Alternative):**
```
TF(t, d) = 1 + log(count(t, d))
```
- Uses logarithm to dampen the effect of very frequent terms
- Common in production systems

**Example:**
```
Document: "The space adventure movie is about space exploration"
- "space" appears 2 times
- Total words: 8
- TF("space", document) = 2/8 = 0.25 (normalized)
```

#### 2. Inverse Document Frequency (IDF)

**IDF** measures how rare or common a term is across the entire corpus (collection of documents). Rare words are more informative!

**Standard IDF Formula:**
```
IDF(t, D) = log(N / df(t, D))
```
- `N` = total number of documents in the corpus
- `df(t, D)` = document frequency (number of documents containing term `t`)
- `log` = natural logarithm (or log base 10)

**Why the logarithm?**
- Prevents very rare words from having extremely high IDF values
- Smooths the scaling

**Smooth IDF (Add-1 Smoothing - More Common):**
```
IDF(t, D) = log((N + 1) / (df(t, D) + 1)) + 1
```
- Prevents division by zero if a term doesn't appear in any document
- Adds 1 to avoid taking log of 0
- This is what scikit-learn uses by default!

**Example:**
```
Corpus: 1000 documents
- "the" appears in 950 documents → IDF = log(1000/950) ≈ 0.05 (very low - common word)
- "space" appears in 50 documents → IDF = log(1000/50) ≈ 3.0 (high - rare word)
- "python" appears in 10 documents → IDF = log(1000/10) ≈ 4.6 (very high - very rare)
```

#### 3. TF-IDF (Combined)

**Final TF-IDF Score:**
```
TF-IDF(t, d, D) = TF(t, d) × IDF(t, D)
```

**What this means:**
- **High TF-IDF**: Term appears frequently in the document AND is rare across the corpus
- **Low TF-IDF**: Term is common everywhere (like "the", "a") OR doesn't appear in the document

**Example Calculation:**
```
Document: "The space adventure movie is about space exploration"
Corpus: 1000 documents

For term "space":
- TF("space", document) = 2/8 = 0.25
- df("space", corpus) = 50 (appears in 50 documents)
- IDF("space", corpus) = log(1000/50) ≈ 3.0
- TF-IDF("space", document) = 0.25 × 3.0 = 0.75

For term "the":
- TF("the", document) = 1/8 = 0.125
- df("the", corpus) = 950 (appears in 950 documents)
- IDF("the", corpus) = log(1000/950) ≈ 0.05
- TF-IDF("the", document) = 0.125 × 0.05 = 0.00625 (very low!)
```

**Why it works**: 
- Common words like "the", "a" have high TF but **very low IDF** (they appear in many documents), so their TF-IDF is low
- Important words like "Python", "space", "adventure" have **high TF-IDF** because they're both frequent in the document AND rare across the corpus
- This automatically filters out stop words and highlights important content words!

### TF-IDF Vector Representation

Each document becomes a vector where:
- **Each dimension** = one word in the vocabulary
- **Each value** = TF-IDF score for that word in that document
- **Most values are 0** (sparse vector) - most words don't appear in each document

**Example:**
```
Vocabulary: ["adventure", "exploration", "movie", "space", "the", ...]

Document 1: "space adventure"
→ Vector: [0.5, 0.0, 0.0, 0.7, 0.0, ...]

Document 2: "movie exploration"
→ Vector: [0.0, 0.6, 0.4, 0.0, 0.0, ...]
```

**Important**: You'll implement TF-IDF from scratch in the **Exercise Notebook**! Here we'll just create TF-IDF vectors together (using scikit-learn) so we can use them for similarity search and clustering. The exercise will teach you how TF-IDF actually works step-by-step.


In [ ]:
# TF-IDF Vectors - Let's create them together using scikit-learn!
# NOTE: You'll implement TF-IDF from scratch in Exercise 3!
# Here we'll use scikit-learn to create TF-IDF vectors for similarity search and clustering

# TODO (Together): Create TF-IDF vectorizer
# What parameters should we use?
vectorizer = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)  # TODO: Discuss these parameters

# TODO (Together): Fit and transform all documents
# What does 'fit' do? What does 'transform' do?
tfidf_vectors = vectorizer.fit_transform(df['description'])  # TODO: Complete this

print(f"TF-IDF Matrix Shape: {tfidf_vectors.shape}")
print(f"(Number of documents, Vocabulary size)")
print(f"\nVocabulary (first 20 words):")
print(vectorizer.get_feature_names_out()[:20])

print(f"\n💡 Remember: You'll learn to implement TF-IDF step-by-step in Exercise 3!")
print(f"   For now, we're using scikit-learn to create vectors for similarity search and clustering.")


## Similarity-Based Search with TF-IDF

**Now let's implement similarity-based search together!** This finds relevant documents based on similarity, not just exact keyword matches.

The idea:
1. Convert query to TF-IDF vector (using the same vectorizer)
2. Compare with all document vectors using **Cosine Similarity**
3. Return most similar documents (highest similarity scores)

**Cosine Similarity**: Measures the angle between two vectors. Range: -1 to 1
- **1** = identical direction (very similar)
- **0** = perpendicular (no similarity)
- **-1** = opposite direction (very different)

**Why cosine?** It measures similarity regardless of document length!


In [ ]:
# Similarity-Based Search - Let's implement this together!
def search_tfidf(query, vectorizer, tfidf_vectors, df, top_k=5):
    """
    Similarity-based search using TF-IDF and cosine similarity
    
    Steps:
    1. Convert query to TF-IDF vector
    2. Calculate cosine similarity with all documents
    3. Get top_k most similar documents
    4. Return results as DataFrame
    """
    # TODO (Together): Step 1 - Convert query to TF-IDF vector using the same vectorizer
    # What method should we use? transform() or fit_transform()?
    query_vector = vectorizer.transform([query])  # TODO: Complete this
    
    # TODO (Together): Step 2 - Calculate cosine similarity with all document vectors
    # How do we compare query_vector with tfidf_vectors?
    similarities = cosine_similarity(query_vector, tfidf_vectors)[0]  # TODO: Complete this
    
    # TODO (Together): Step 3 - Get indices of top_k most similar documents
    # How do we find the indices of the highest similarity scores?
    # Hint: Use argsort() and reverse order
    top_indices = similarities.argsort()[-top_k:][::-1]  # TODO: Complete this
    
    # TODO (Together): Step 4 - Build results DataFrame
    results = []
    for idx in top_indices:
        results.append({
            'movie_id': df.iloc[idx]['movie_id'],
            'title': df.iloc[idx]['title'],
            'similarity': similarities[idx],
            'description': df.iloc[idx]['description']
        })
    
    return pd.DataFrame(results)

# Let's test it!
example_query = "space exploration adventure"
print(f"Query: '{example_query}'")
print("\nResults:")
results = search_tfidf(example_query, vectorizer, tfidf_vectors, df, top_k=5)
print(results[['title', 'similarity']])


### Compare: Keyword Search vs Similarity-Based Search (TF-IDF)

Let's see how they differ:

**Note**: TF-IDF similarity search is better than simple keyword matching, but it's still fundamentally keyword-based (not true semantic search). True semantic search that understands synonyms and meaning requires embeddings (Class 3)!


In [ ]:
# Comparison: Keyword Search vs Similarity-Based Search
# You'll implement both in exercises!

print("=" * 60)
print("KEY DIFFERENCES: Keyword Search vs Similarity Search")
print("=" * 60)

print("\nKeyword Search (from Part 1 - Exercise 4):")
print("  ✅ Fast and simple")
print("  ✅ Finds exact word matches")
print("  ✅ Multiple keyword search allows more specific queries")
print("  ❌ No ranking - all matches are equal")
print("  ❌ 'mind-bending' won't find 'psychological thriller'")
print("  ❌ Limited to exact words")

print("\nSimilarity-Based Search (TF-IDF - Exercise 5):")
print("  ✅ Ranks by importance (TF-IDF weighted)")
print("  ✅ Better than keyword search")
print("  ✅ Finds documents with similar word patterns")
print("  ⚠️ Still keyword-based (not true semantic)")
print("  ⚠️ 'mind-bending' might match 'psychological' IF they share other words")
print("  ❌ Still cannot understand synonyms or true meaning")

print("\nTrue Semantic Search (Embeddings - Class 3!):")
print("  ✅ Understands meaning and synonyms")
print("  ✅ 'space' and 'cosmic' are similar (semantic similarity)")
print("  ✅ True understanding of meaning")
print("  ⚠️ Requires embeddings (Class 3)")

print("\n💡 Key Point: TF-IDF is BETTER than keyword search,")
print("   but it's still SYNTAX-based (word patterns), not SEMANTIC (meaning).")
print("   Semantic = meaning. True semantic search requires embeddings!")

print("\n💡 Implementation:")
print("   - Exercise 4: Keyword search")
print("   - Exercise 5: Similarity-based search with TF-IDF")
print("   - Class 3: Semantic search with embeddings")


## Hybrid Search: Combining TF-IDF with Keyword Matching

**Real-world search systems** often combine multiple approaches to get the best results. Let's build a **hybrid search** that combines:
1. **TF-IDF similarity search** (semantic-like matching, ranks by importance)
2. **Keyword matching** (exact word matches, boosts precision)

**Why Hybrid?**
- **TF-IDF alone**: Might miss exact matches or rank them lower
- **Keyword alone**: Too rigid, misses related content
- **Hybrid**: Best of both worlds - precision from keywords + recall from TF-IDF!

### Hybrid Search Strategy

**Approach**: Combine scores from both methods
1. Get TF-IDF similarity scores (0 to 1)
2. Get keyword match scores (binary: 0 or 1, or count-based)
3. Combine with weighted sum: `final_score = α × keyword_score + (1-α) × tfidf_score`
   - `α` (alpha) = weight for keyword matching (e.g., 0.3 = 30% keyword, 70% TF-IDF)

**Alternative**: Boost exact matches
- If query words appear in document → boost TF-IDF score
- Simple: `final_score = tfidf_score + boost × keyword_match_count`

Let's implement a hybrid search for our movie search system!


In [ ]:
# Hybrid Search Implementation - Combining TF-IDF + Keyword Matching
def hybrid_search(query, vectorizer, tfidf_vectors, df, top_k=5, keyword_weight=0.3, boost_factor=0.2):
    """
    Hybrid search combining TF-IDF similarity and keyword matching
    
    Args:
        query: Search query string
        vectorizer: Fitted TF-IDF vectorizer
        tfidf_vectors: TF-IDF vectors for all documents
        df: DataFrame with movie data
        top_k: Number of results to return
        keyword_weight: Weight for keyword matching (0-1). 0.3 = 30% keyword, 70% TF-IDF
        boost_factor: Additional boost for exact keyword matches (added to TF-IDF score)
    
    Returns:
        DataFrame with search results sorted by combined score
    """
    import re
    
    # Step 1: Get TF-IDF similarity scores
    query_vector = vectorizer.transform([query])
    tfidf_similarities = cosine_similarity(query_vector, tfidf_vectors)[0]
    
    # Step 2: Get keyword match scores
    query_words = set(re.findall(r'\w+', query.lower()))  # Extract words from query
    keyword_scores = []
    keyword_match_counts = []
    
    for idx, row in df.iterrows():
        text = str(row['description']).lower()
        text_words = set(re.findall(r'\w+', text))
        
        # Count how many query words appear in document
        matches = query_words.intersection(text_words)
        match_count = len(matches)
        match_ratio = match_count / len(query_words) if len(query_words) > 0 else 0
        
        keyword_scores.append(match_ratio)  # Normalized: 0 to 1
        keyword_match_counts.append(match_count)  # Raw count
    
    keyword_scores = np.array(keyword_scores)
    keyword_match_counts = np.array(keyword_match_counts)
    
    # Step 3: Combine scores using weighted sum
    # Normalize TF-IDF scores to 0-1 range (they're already in that range, but ensure it)
    tfidf_normalized = (tfidf_similarities - tfidf_similarities.min()) / (tfidf_similarities.max() - tfidf_similarities.min() + 1e-8)
    
    # Weighted combination
    combined_scores = (keyword_weight * keyword_scores) + ((1 - keyword_weight) * tfidf_normalized)
    
    # Step 4: Add boost for exact keyword matches
    # Documents with more keyword matches get a boost
    max_matches = keyword_match_counts.max() if keyword_match_counts.max() > 0 else 1
    keyword_boost = (keyword_match_counts / max_matches) * boost_factor
    final_scores = combined_scores + keyword_boost
    
    # Step 5: Get top_k results
    top_indices = final_scores.argsort()[-top_k:][::-1]
    
    # Build results
    results = []
    for idx in top_indices:
        results.append({
            'movie_id': df.iloc[idx]['movie_id'],
            'title': df.iloc[idx]['title'],
            'final_score': final_scores[idx],
            'tfidf_score': tfidf_similarities[idx],
            'keyword_score': keyword_scores[idx],
            'keyword_matches': keyword_match_counts[idx],
            'description': df.iloc[idx]['description']
        })
    
    return pd.DataFrame(results)

# Test hybrid search
print("=" * 70)
print("Hybrid Search Example")
print("=" * 70)

query = "space adventure"
print(f"\nQuery: '{query}'")
print(f"\nResults (showing scores breakdown):\n")

results_hybrid = hybrid_search(query, vectorizer, tfidf_vectors, df, top_k=5, 
                               keyword_weight=0.3, boost_factor=0.2)

print(results_hybrid[['title', 'final_score', 'tfidf_score', 'keyword_score', 'keyword_matches']].to_string(index=False))

print("\n" + "=" * 70)
print("Understanding the Scores:")
print("=" * 70)
print("  - final_score: Combined score (keyword + TF-IDF + boost)")
print("  - tfidf_score: TF-IDF cosine similarity (0-1)")
print("  - keyword_score: Ratio of query words found (0-1)")
print("  - keyword_matches: Number of query words that appear in document")
print("\n💡 Documents with exact keyword matches get boosted!")
print("💡 But TF-IDF still helps find related content even without exact matches!")


### Comparing Search Methods: Keyword vs TF-IDF vs Hybrid

Let's see how different search approaches perform on the same query:


In [ ]:
# Compare: Keyword Search vs TF-IDF vs Hybrid Search
def simple_keyword_search_comparison(query, df, top_k=5):
    """Simple keyword search for comparison"""
    import re
    query_words = set(re.findall(r'\w+', query.lower()))
    results = []
    
    for idx, row in df.iterrows():
        text = str(row['description']).lower()
        text_words = set(re.findall(r'\w+', text))
        matches = query_words.intersection(text_words)
        match_count = len(matches)
        
        if match_count > 0:
            results.append({
                'movie_id': row['movie_id'],
                'title': row['title'],
                'match_count': match_count,
                'description': row['description']
            })
    
    # Sort by match count (descending)
    results_df = pd.DataFrame(results)
    if len(results_df) > 0:
        results_df = results_df.sort_values('match_count', ascending=False).head(top_k)
    return results_df

# Test all three methods
test_query = "space adventure"

print("=" * 80)
print(f"COMPARISON: Search Results for Query '{test_query}'")
print("=" * 80)

# Method 1: Keyword Search
print("\n" + "=" * 80)
print("1. KEYWORD SEARCH (Exact Word Matching)")
print("=" * 80)
keyword_results = simple_keyword_search_comparison(test_query, df, top_k=5)
if len(keyword_results) > 0:
    print(keyword_results[['title', 'match_count']].to_string(index=False))
    print(f"\n✅ Finds documents with exact word matches")
    print(f"❌ No ranking beyond match count")
    print(f"❌ Misses related content without exact words")
else:
    print("No results found")

# Method 2: TF-IDF Search
print("\n" + "=" * 80)
print("2. TF-IDF SIMILARITY SEARCH")
print("=" * 80)
tfidf_results = search_tfidf(test_query, vectorizer, tfidf_vectors, df, top_k=5)
print(tfidf_results[['title', 'similarity']].to_string(index=False))
print(f"\n✅ Ranks by importance (TF-IDF weighted)")
print(f"✅ Finds related content even without exact matches")
print(f"⚠️  Might rank exact matches lower if they have low TF-IDF")

# Method 3: Hybrid Search
print("\n" + "=" * 80)
print("3. HYBRID SEARCH (TF-IDF + Keyword Matching)")
print("=" * 80)
hybrid_results = hybrid_search(test_query, vectorizer, tfidf_vectors, df, top_k=5, 
                               keyword_weight=0.3, boost_factor=0.2)
print(hybrid_results[['title', 'final_score', 'tfidf_score', 'keyword_matches']].to_string(index=False))
print(f"\n✅ Combines best of both: keyword precision + TF-IDF recall")
print(f"✅ Boosts exact matches while still finding related content")
print(f"✅ Most flexible and robust approach")

print("\n" + "=" * 80)
print("SUMMARY: When to Use Each Method")
print("=" * 80)
print("""
Keyword Search:
  - ✅ Fast and simple
  - ✅ Good for exact phrase matching
  - ❌ Too rigid, misses related content
  - Use when: You need exact matches only

TF-IDF Search:
  - ✅ Better ranking by importance
  - ✅ Finds related content
  - ⚠️  Might miss exact matches
  - Use when: You want semantic-like matching (but still keyword-based)

Hybrid Search:
  - ✅ Best of both worlds
  - ✅ Precision from keywords + recall from TF-IDF
  - ✅ Most robust for production systems
  - Use when: Building a real search system (recommended!)
""")

print("💡 For our movie search system, hybrid search gives the best results!")


### Tuning Hybrid Search Parameters

**Key Parameters to Adjust:**

1. **`keyword_weight`** (0 to 1):
   - **0.0** = Pure TF-IDF search (no keyword boost)
   - **0.3-0.4** = Balanced (recommended starting point)
   - **0.5-0.7** = More emphasis on exact keyword matches
   - **1.0** = Pure keyword search (no TF-IDF)

2. **`boost_factor`** (0 to 1):
   - **0.0** = No additional boost for keyword matches
   - **0.1-0.3** = Moderate boost (recommended)
   - **0.5+** = Strong boost (may over-prioritize exact matches)

**Tuning Strategy:**
- Start with `keyword_weight=0.3` and `boost_factor=0.2`
- Test on sample queries and check if results make sense
- If exact matches are too low → increase `keyword_weight` or `boost_factor`
- If results are too rigid → decrease `keyword_weight`

**Example: Different Configurations**

```python
# Configuration 1: Balanced (default)
hybrid_search(query, ..., keyword_weight=0.3, boost_factor=0.2)

# Configuration 2: More emphasis on exact matches
hybrid_search(query, ..., keyword_weight=0.5, boost_factor=0.3)

# Configuration 3: More TF-IDF, less keyword
hybrid_search(query, ..., keyword_weight=0.2, boost_factor=0.1)
```

**💡 Pro Tip**: In production, you might want to:
- Use A/B testing to find optimal parameters
- Different weights for different query types (short vs long queries)
- User feedback to continuously improve the search quality


---

## 🎯 Interactive Exercise: Visualizing Sentences in Vector Space

**Goal**: Experiment with different vectorization methods and preprocessing to see how they affect the vector space!

**What you'll do:**
1. Create your own sentences (similar and dissimilar)
2. **Experiment** with different vectorization methods:
   - **Bag of Words (BoW)**: Simple word counts
   - **TF-IDF**: Weighted word frequencies
3. **Experiment** with preprocessing:
   - **With preprocessing**: Cleaned, normalized text
   - **Without preprocessing**: Raw text
4. Visualize them in 2D space using PCA (for visualization only)
5. Compare how different methods affect clustering and distances

**Why this matters**: Understanding how different choices affect vector space helps you make better decisions in real projects!

**💡 Be Curious!** Try different combinations and see what happens:
- Does preprocessing make sentences cluster better?
- How does BoW compare to TF-IDF?
- Which method works best for your sentences?

**Note**: PCA (Principal Component Analysis) is used here **only for visualization** - it reduces high-dimensional vectors to 2D so we can plot them. We'll explain PCA briefly, but the focus is on understanding vector space, not PCA itself.


### Understanding PCA for Visualization (Brief Explanation)

**The Problem**: TF-IDF vectors are high-dimensional (100+ dimensions). We can't visualize 100D space!

**The Solution**: PCA (Principal Component Analysis) projects high-dimensional vectors to 2D for visualization.

**What PCA does**:
- Finds the 2 most important directions (principal components) in the high-dimensional space
- Projects all vectors onto these 2 directions
- Result: 2D coordinates we can plot!

**Important**: 
- PCA is **only for visualization** - we still use full TF-IDF vectors for search/clustering
- Some information is lost in 2D projection, but the main patterns are preserved
- Similar sentences will still appear close together in 2D (if they're close in high-D space)

**Variance Explained**: Tells us how much information is preserved. 80% variance = 80% of the information is kept in 2D.

Let's visualize sentences in 2D space!


In [ ]:
# Flexible Visualization Helper Function - Experiment with Different Methods!
def visualize_sentences_in_space(custom_sentences, corpus_texts, 
                                  method='tfidf', use_preprocessing=True,
                                  corpus_labels=None, sample_size=50, 
                                  title=None):
    """
    Visualize custom sentences and corpus documents in 2D space using PCA.
    Supports different vectorization methods and preprocessing options!
    
    Args:
        custom_sentences: List of custom sentences to visualize
        corpus_texts: List of corpus text descriptions
        method: 'tfidf' or 'bow' (Bag of Words)
        use_preprocessing: True to use preprocessing (stop words, lowercase, etc.), False for raw
        corpus_labels: Optional labels for corpus points (e.g., movie titles)
        sample_size: Number of corpus documents to show (for clarity)
        title: Plot title (auto-generated if None)
    """
    from sklearn.decomposition import PCA
    from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
    import matplotlib.pyplot as plt
    from scipy.sparse import vstack
    import random
    
    # Generate title if not provided
    if title is None:
        method_name = "TF-IDF" if method == 'tfidf' else "Bag of Words"
        prep_name = "with preprocessing" if use_preprocessing else "without preprocessing"
        title = f"Sentences in Vector Space ({method_name}, {prep_name})"
    
    # Step 1: Choose vectorizer based on method and preprocessing
    if method.lower() == 'tfidf':
        if use_preprocessing:
            vectorizer = TfidfVectorizer(max_features=100, stop_words='english', 
                                        lowercase=True, token_pattern=r'\b\w+\b')
        else:
            vectorizer = TfidfVectorizer(max_features=100, lowercase=False, 
                                        token_pattern=r'\b\w+\b')
    else:  # bow
        if use_preprocessing:
            vectorizer = CountVectorizer(max_features=100, stop_words='english', 
                                        lowercase=True, token_pattern=r'\b\w+\b')
        else:
            vectorizer = CountVectorizer(max_features=100, lowercase=False, 
                                        token_pattern=r'\b\w+\b')
    
    # Step 2: Fit vectorizer on corpus and transform
    print(f"🔧 Using: {method.upper()} {'with' if use_preprocessing else 'without'} preprocessing")
    corpus_vectors = vectorizer.fit_transform(corpus_texts)
    custom_vectors = vectorizer.transform(custom_sentences)
    
    # Step 3: Sample corpus vectors (if too many, sample for clarity)
    if corpus_vectors.shape[0] > sample_size:
        random.seed(42)
        sample_indices = random.sample(range(corpus_vectors.shape[0]), sample_size)
        corpus_vectors_sample = corpus_vectors[sample_indices]
        if corpus_labels is not None:
            corpus_labels_sample = [corpus_labels[i] for i in sample_indices]
        else:
            corpus_labels_sample = None
    else:
        corpus_vectors_sample = corpus_vectors
        corpus_labels_sample = corpus_labels
    
    # Step 4: Combine custom and corpus vectors
    all_vectors = vstack([custom_vectors, corpus_vectors_sample])
    
    # Step 5: Convert to dense (PCA requires dense arrays)
    all_vectors_dense = all_vectors.toarray()
    
    # Step 6: Apply PCA to reduce to 2D
    pca = PCA(n_components=2, random_state=42)
    vectors_2d = pca.fit_transform(all_vectors_dense)
    
    # Step 7: Split back into custom and corpus
    n_custom = len(custom_sentences)
    custom_2d = vectors_2d[:n_custom]
    corpus_2d = vectors_2d[n_custom:]
    
    # Step 8: Plot
    plt.figure(figsize=(12, 8))
    
    # Plot corpus points (small, gray)
    plt.scatter(corpus_2d[:, 0], corpus_2d[:, 1], 
                c='lightgray', alpha=0.5, s=30, label='Corpus documents')
    
    # Plot custom sentences (large, colored)
    colors = plt.cm.tab10(range(n_custom))
    for i, (x, y) in enumerate(custom_2d):
        plt.scatter(x, y, c=[colors[i]], s=200, alpha=0.7, 
                   edgecolors='black', linewidths=2, zorder=5)
        # Add label
        plt.annotate(f'S{i+1}', (x, y), 
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=10, fontweight='bold', zorder=6)
    
    plt.xlabel(f'First Principal Component (explains {pca.explained_variance_ratio_[0]:.1%} variance)')
    plt.ylabel(f'Second Principal Component (explains {pca.explained_variance_ratio_[1]:.1%} variance)')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print variance explained
    total_variance = pca.explained_variance_ratio_.sum()
    print(f"📊 Total variance explained by 2D projection: {total_variance:.1%}")
    print(f"   (This means {total_variance:.1%} of the information is preserved in 2D)")
    
    # Print custom sentences with their positions
    print("\n" + "=" * 70)
    print("Your Custom Sentences:")
    print("=" * 70)
    for i, sentence in enumerate(custom_sentences):
        print(f"\nSentence {i+1}: '{sentence}'")
        print(f"  2D Position: ({custom_2d[i, 0]:.2f}, {custom_2d[i, 1]:.2f})")
    
    return custom_2d, corpus_2d, pca, vectorizer

print("✅ Flexible visualization function ready!")
print("💡 You can now experiment with:")
print("   - method='tfidf' or method='bow'")
print("   - use_preprocessing=True or False")


### Exercise 1: Create Similar and Dissimilar Sentences

**Your Task**: Create sentences that are:
- **Similar** to each other (should appear close in vector space)
- **Dissimilar** to each other (should appear far apart in vector space)

**Tips for creating similar sentences**:
- Use similar words and topics
- Example: "space adventure movie" and "cosmic journey film" (similar topic, different words)

**Tips for creating dissimilar sentences**:
- Use completely different topics
- Example: "space adventure movie" and "cooking recipe book" (completely different)

Let's try it!


In [ ]:
# TODO: Create your own sentences and experiment with different methods!
# Try to create:
# - 2-3 similar sentences (should appear close together in space)
# - 1-2 dissimilar sentences (should appear far from the similar ones)

# Example sentences (modify these or create your own!)
custom_sentences = [
    # Similar sentences (space/adventure theme)
    "space adventure movie with aliens",
    "cosmic journey film about exploration",
    "galactic quest story in outer space",
    
    # Dissimilar sentence (completely different topic)
    "cooking recipe book for desserts",
    # Add more dissimilar sentences here!
]

print("Your custom sentences:")
for i, sent in enumerate(custom_sentences, 1):
    print(f"  {i}. {sent}")

# 🎯 EXPERIMENT: Try different combinations!
# Change these parameters and see what happens:
METHOD = 'tfidf'  # Try: 'tfidf' or 'bow'
USE_PREPROCESSING = True  # Try: True or False

print("\n" + "=" * 70)
print(f"🔬 Experiment: {METHOD.upper()} {'with' if USE_PREPROCESSING else 'without'} preprocessing")
print("=" * 70)

# Visualize them in vector space!
custom_2d, corpus_2d, pca, vec = visualize_sentences_in_space(
    custom_sentences=custom_sentences,
    corpus_texts=df['description'].tolist(),
    method=METHOD,
    use_preprocessing=USE_PREPROCESSING,
    corpus_labels=df['title'].tolist(),
    sample_size=50
)

# Calculate distances between sentences
print("\n" + "=" * 70)
print("Distances Between Your Sentences (in 2D space):")
print("=" * 70)
from scipy.spatial.distance import euclidean

for i in range(len(custom_sentences)):
    for j in range(i+1, len(custom_sentences)):
        dist = euclidean(custom_2d[i], custom_2d[j])
        print(f"\nDistance between:")
        print(f"  '{custom_sentences[i]}'")
        print(f"  '{custom_sentences[j]}'")
        print(f"  → Distance: {dist:.2f} (smaller = more similar)")

print("\n💡 Observations:")
print("  - Similar sentences should have SMALL distances")
print("  - Dissimilar sentences should have LARGE distances")
print("  - Check if your similar sentences are close together!")

print("\n" + "=" * 70)
print("🎓 Try This:")
print("=" * 70)
print("1. Run this cell again with METHOD='bow' - how does it compare?")
print("2. Try USE_PREPROCESSING=False - do sentences cluster differently?")
print("3. Which combination works best for your sentences?")
print("4. Notice how preprocessing affects the vocabulary and clustering!")


### Exercise 2: Find Similar Sentences from the Corpus

**Your Task**: Create a sentence and find the most similar sentences from the movie corpus!

This exercise helps you understand:
- How TF-IDF similarity works in practice
- Which movies are most similar to your query
- How similarity scores relate to vector space positions


In [ ]:
# TODO: Create your query sentence and experiment with different methods!
# Try different queries and see which movies are most similar!

my_query = "space adventure with aliens and exploration"  # Modify this!

# 🎯 EXPERIMENT: Try different vectorization methods!
METHOD = 'tfidf'  # Try: 'tfidf' or 'bow'
USE_PREPROCESSING = True  # Try: True or False

print("=" * 70)
print(f"Finding movies similar to: '{my_query}'")
print(f"Using: {METHOD.upper()} {'with' if USE_PREPROCESSING else 'without'} preprocessing")
print("=" * 70)

# Create vectorizer based on method
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
if METHOD == 'tfidf':
    if USE_PREPROCESSING:
        vec = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = TfidfVectorizer(max_features=100, lowercase=False)
else:  # bow
    if USE_PREPROCESSING:
        vec = CountVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = CountVectorizer(max_features=100, lowercase=False)

# Fit and transform corpus
corpus_vecs = vec.fit_transform(df['description'])
query_vec = vec.transform([my_query])

# Calculate similarities
from sklearn.metrics.pairwise import cosine_similarity
similarities = cosine_similarity(query_vec, corpus_vecs)[0]

# Get top 5
top_indices = similarities.argsort()[-5:][::-1]
similar_movies = pd.DataFrame({
    'title': [df.iloc[i]['title'] for i in top_indices],
    'similarity': [similarities[i] for i in top_indices],
    'description': [df.iloc[i]['description'] for i in top_indices]
})

print("\nTop 5 Most Similar Movies:")
print(similar_movies[['title', 'similarity']].to_string(index=False))

# Visualize: Show your query and the most similar movies in space
print("\n" + "=" * 70)
print("Visualizing your query and similar movies in vector space...")
print("=" * 70)

# Get top 3 similar movie descriptions
top_movie_descriptions = similar_movies.head(3)['description'].tolist()
sentences_to_visualize = [my_query] + top_movie_descriptions

# Visualize
custom_2d, corpus_2d, pca, _ = visualize_sentences_in_space(
    custom_sentences=sentences_to_visualize,
    corpus_texts=df['description'].tolist(),
    method=METHOD,
    use_preprocessing=USE_PREPROCESSING,
    corpus_labels=df['title'].tolist(),
    sample_size=50
)

# Show distances
print("\n" + "=" * 70)
print("Distances from Your Query to Similar Movies:")
print("=" * 70)
from scipy.spatial.distance import euclidean

query_pos = custom_2d[0]  # First sentence is the query
for i, movie_title in enumerate(similar_movies.head(3)['title']):
    movie_pos = custom_2d[i+1]  # +1 because first is query
    dist = euclidean(query_pos, movie_pos)
    similarity_score = similar_movies.iloc[i]['similarity']
    print(f"\n{movie_title}:")
    print(f"  Distance in 2D: {dist:.2f}")
    print(f"  Similarity Score: {similarity_score:.3f}")
    print(f"  → Lower distance = closer in space = more similar!")

print("\n💡 Key Insight:")
print("  - Movies with HIGH similarity scores should be CLOSE to your query in the plot")
print("  - Movies with LOW similarity scores should be FAR from your query")
print("  - This shows how vectorization represents similarity as distance in vector space!")

print("\n" + "=" * 70)
print("🎓 Try This:")
print("=" * 70)
print("1. Try METHOD='bow' - do you get different similar movies?")
print("2. Try USE_PREPROCESSING=False - how does it affect results?")
print("3. Which method finds the most relevant movies for your query?")


### Exercise 3: Observe Natural Clustering

**Your Task**: Create sentences from different topics and observe how they naturally form clusters in vector space!

This exercise helps you understand:
- How similar sentences naturally group together (clustering)
- Why clustering algorithms work - similar items are close in space
- How different topics form separate clusters


In [ ]:
# TODO: Create sentences from 2-3 different topics and experiment!
# Each topic should have 2-3 similar sentences
# The goal is to see how sentences from the same topic cluster together!

# Example: Create sentences from different topics
topic_sentences = {
    "Space/Adventure": [
        "space adventure movie with aliens",
        "cosmic journey film about exploration",
        "galactic quest story in outer space"
    ],
    "Romance": [
        "romantic love story between two people",
        "heartwarming tale of romance and relationships",
        "emotional drama about love and connection"
    ],
    "Action": [
        "action movie with intense fight scenes",
        "thrilling adventure with explosions and chases",
        "high-energy film with combat and stunts"
    ],
    # Add your own topic here!
    # "Your Topic": [
    #     "sentence 1 about your topic",
    #     "sentence 2 about your topic",
    # ]
}

# Flatten into a list
all_sentences = []
topic_labels = []
for topic, sentences in topic_sentences.items():
    all_sentences.extend(sentences)
    topic_labels.extend([topic] * len(sentences))

print("Sentences by topic:")
for topic, sentences in topic_sentences.items():
    print(f"\n{topic}:")
    for sent in sentences:
        print(f"  - {sent}")

# 🎯 EXPERIMENT: Try different methods and see how clustering changes!
METHOD = 'tfidf'  # Try: 'tfidf' or 'bow'
USE_PREPROCESSING = True  # Try: True or False

# Visualize with color coding by topic
print("\n" + "=" * 70)
print(f"🔬 Experiment: {METHOD.upper()} {'with' if USE_PREPROCESSING else 'without'} preprocessing")
print("Visualizing sentences grouped by topic...")
print("=" * 70)

from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
import matplotlib.pyplot as plt
from scipy.sparse import vstack
import random

# Create vectorizer based on method
if METHOD == 'tfidf':
    if USE_PREPROCESSING:
        vec = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = TfidfVectorizer(max_features=100, lowercase=False)
else:  # bow
    if USE_PREPROCESSING:
        vec = CountVectorizer(max_features=100, stop_words='english', lowercase=True)
    else:
        vec = CountVectorizer(max_features=100, lowercase=False)

# Convert to vectors
corpus_vecs = vec.fit_transform(df['description'])
sentence_vectors = vec.transform(all_sentences)

# Sample corpus for background
if corpus_vecs.shape[0] > 50:
    random.seed(42)
    sample_indices = random.sample(range(corpus_vecs.shape[0]), 50)
    corpus_sample = corpus_vecs[sample_indices]
else:
    corpus_sample = corpus_vecs

# Combine and apply PCA
from scipy.sparse import vstack
all_vectors = vstack([sentence_vectors, corpus_sample])
all_vectors_dense = all_vectors.toarray()

pca = PCA(n_components=2, random_state=42)
vectors_2d = pca.fit_transform(all_vectors_dense)

n_sentences = len(all_sentences)
sentence_2d = vectors_2d[:n_sentences]
corpus_2d = vectors_2d[n_sentences:]

# Plot with colors by topic
plt.figure(figsize=(14, 10))

# Plot corpus background
plt.scatter(corpus_2d[:, 0], corpus_2d[:, 1], 
            c='lightgray', alpha=0.3, s=20, label='Corpus documents')

# Plot sentences colored by topic
unique_topics = list(topic_sentences.keys())
colors = plt.cm.Set3(range(len(unique_topics)))
topic_to_color = {topic: colors[i] for i, topic in enumerate(unique_topics)}

for topic in unique_topics:
    topic_indices = [i for i, t in enumerate(topic_labels) if t == topic]
    topic_positions = sentence_2d[topic_indices]
    plt.scatter(topic_positions[:, 0], topic_positions[:, 1],
               c=[topic_to_color[topic]], s=200, alpha=0.7,
               edgecolors='black', linewidths=2, label=topic, zorder=5)
    
    # Add labels
    for idx in topic_indices:
        plt.annotate(f"S{idx+1}", sentence_2d[idx],
                    xytext=(5, 5), textcoords='offset points',
                    fontsize=9, fontweight='bold', zorder=6)

plt.xlabel(f'First Principal Component (explains {pca.explained_variance_ratio_[0]:.1%} variance)')
plt.ylabel(f'Second Principal Component (explains {pca.explained_variance_ratio_[1]:.1%} variance)')
plt.title("Natural Clustering: Sentences Grouped by Topic")
plt.legend(loc='best')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Calculate average distances within and between topics
print("\n" + "=" * 70)
print("Clustering Analysis:")
print("=" * 70)

from scipy.spatial.distance import euclidean

for topic in unique_topics:
    topic_indices = [i for i, t in enumerate(topic_labels) if t == topic]
    topic_positions = sentence_2d[topic_indices]
    
    # Average distance within topic
    within_distances = []
    for i in range(len(topic_indices)):
        for j in range(i+1, len(topic_indices)):
            dist = euclidean(topic_positions[i], topic_positions[j])
            within_distances.append(dist)
    
    avg_within = np.mean(within_distances) if within_distances else 0
    print(f"\n{topic}:")
    print(f"  Average distance within topic: {avg_within:.2f}")
    print(f"  (Lower = sentences are closer together = better cluster)")

# Average distance between topics
print("\n" + "=" * 70)
print("Between-Topic Distances:")
print("=" * 70)

for i, topic1 in enumerate(unique_topics):
    for topic2 in unique_topics[i+1:]:
        indices1 = [i for i, t in enumerate(topic_labels) if t == topic1]
        indices2 = [i for i, t in enumerate(topic_labels) if t == topic2]
        
        between_distances = []
        for idx1 in indices1:
            for idx2 in indices2:
                dist = euclidean(sentence_2d[idx1], sentence_2d[idx2])
                between_distances.append(dist)
        
        avg_between = np.mean(between_distances) if between_distances else 0
        print(f"\n{topic1} ↔ {topic2}:")
        print(f"  Average distance: {avg_between:.2f}")
        print(f"  (Higher = topics are more separated = better clustering)")

print("\n💡 Key Observations:")
print("  ✅ Sentences from the SAME topic should be CLOSE together (low within-topic distance)")
print("  ✅ Sentences from DIFFERENT topics should be FAR apart (high between-topic distance)")
print("  ✅ This is why clustering algorithms work - similar items are naturally close in space!")
print("  ✅ In the plot, you should see sentences from the same topic grouped together!")

print("\n" + "=" * 70)
print("🎓 Try This:")
print("=" * 70)
print("1. Try METHOD='bow' - do topics cluster better or worse?")
print("2. Try USE_PREPROCESSING=False - how does it affect clustering?")
print("3. Which combination gives the best separation between topics?")
print("4. Notice how preprocessing can help or hurt clustering depending on your data!")
print("5. Experiment with different topics - some might cluster better than others!")


### 🎓 Bonus: Compare All Methods Side-by-Side

Want to see all combinations at once? Try this comparison!


In [ ]:
# 🎯 Compare All Methods Side-by-Side
# This creates 4 plots showing all combinations of methods and preprocessing

# Use the same sentences from Exercise 1 (or create your own!)
test_sentences = [
    "space adventure movie with aliens",
    "cosmic journey film about exploration",
    "cooking recipe book for desserts"
]

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from scipy.sparse import vstack
import random

fig, axes = plt.subplots(2, 2, figsize=(16, 14))
fig.suptitle('Comparing All Methods: How Do They Affect Vector Space?', fontsize=16, fontweight='bold')

configs = [
    ('tfidf', True, 'TF-IDF with preprocessing'),
    ('tfidf', False, 'TF-IDF without preprocessing'),
    ('bow', True, 'BoW with preprocessing'),
    ('bow', False, 'BoW without preprocessing')
]

for idx, (method, use_prep, title) in enumerate(configs):
    ax = axes[idx // 2, idx % 2]
    
    # Create vectorizer
    if method == 'tfidf':
        if use_prep:
            vec = TfidfVectorizer(max_features=100, stop_words='english', lowercase=True)
        else:
            vec = TfidfVectorizer(max_features=100, lowercase=False)
    else:  # bow
        if use_prep:
            vec = CountVectorizer(max_features=100, stop_words='english', lowercase=True)
        else:
            vec = CountVectorizer(max_features=100, lowercase=False)
    
    # Fit and transform
    corpus_vecs = vec.fit_transform(df['description'])
    sentence_vecs = vec.transform(test_sentences)
    
    # Sample corpus
    if corpus_vecs.shape[0] > 50:
        random.seed(42)
        sample_indices = random.sample(range(corpus_vecs.shape[0]), 50)
        corpus_sample = corpus_vecs[sample_indices]
    else:
        corpus_sample = corpus_vecs
    
    # Combine and PCA
    all_vecs = vstack([sentence_vecs, corpus_sample])
    all_vecs_dense = all_vecs.toarray()
    
    pca = PCA(n_components=2, random_state=42)
    vectors_2d = pca.fit_transform(all_vecs_dense)
    
    n_sent = len(test_sentences)
    sent_2d = vectors_2d[:n_sent]
    corpus_2d = vectors_2d[n_sent:]
    
    # Plot
    ax.scatter(corpus_2d[:, 0], corpus_2d[:, 1], 
               c='lightgray', alpha=0.3, s=20, label='Corpus')
    
    colors = plt.cm.tab10(range(n_sent))
    for i, (x, y) in enumerate(sent_2d):
        ax.scatter(x, y, c=[colors[i]], s=200, alpha=0.7, 
                  edgecolors='black', linewidths=2, zorder=5)
        ax.annotate(f'S{i+1}', (x, y), 
                   xytext=(5, 5), textcoords='offset points',
                   fontsize=9, fontweight='bold', zorder=6)
    
    ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})')
    ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})')
    ax.set_title(title, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

print("=" * 70)
print("📊 Comparison Summary:")
print("=" * 70)
print("\nObserve:")
print("  - How do the positions of your sentences change?")
print("  - Which method clusters similar sentences better?")
print("  - Does preprocessing help or hurt clustering?")
print("  - Notice the variance explained - which preserves more information?")
print("\n💡 There's no 'best' method - it depends on your data and goals!")
print("   Experimentation is key to finding what works best for you!")


### Summary: What You Learned from Visualization

**Key Takeaways:**

1. **Vector Space Representation**:
   - Each sentence/document is a point in high-dimensional space
   - Similar sentences are close together (small distance)
   - Dissimilar sentences are far apart (large distance)

2. **TF-IDF Similarity**:
   - Similarity = proximity in vector space
   - Cosine similarity measures the angle between vectors
   - High similarity → close in space → similar meaning (in terms of word patterns)

3. **Natural Clustering**:
   - Similar items naturally group together in space
   - This is why clustering algorithms work!
   - Different topics form separate clusters

4. **PCA for Visualization**:
   - Reduces high-dimensional vectors to 2D for plotting
   - Preserves main patterns (though some information is lost)
   - Used only for visualization, not for actual search/clustering

**Remember**: 
- TF-IDF is still **syntactic** (word-based), not **semantic** (meaning-based)
- True semantic understanding requires embeddings (Class 3!)
- But TF-IDF captures word patterns well, which is why similar sentences cluster together

**Next**: We'll use these concepts for clustering documents automatically!


## Clustering Documents

Now let's group similar movies together using **K-Means Clustering** (unsupervised learning!).

**Goal**: Automatically discover groups of similar documents without labels.


In [ ]:
# Clustering - Let's cluster movies together!
n_clusters = 4  # We know there are roughly 4-5 genres

# TODO (Together): Create KMeans clusterer
# What parameters should we use? (n_clusters, random_state, n_init)
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)  # TODO: Complete this

# TODO (Together): Fit and predict clusters
# What does fit_predict do? How is it different from fit() then predict()?
clusters = kmeans.fit_predict(tfidf_vectors)  # TODO: Complete this

# Add cluster labels to dataframe
df['cluster'] = clusters

# Display clusters
print("Movie Clusters:")
print("=" * 60)
for cluster_id in range(n_clusters):
    cluster_movies = df[df['cluster'] == cluster_id]
    print(f"\nCluster {cluster_id} ({len(cluster_movies)} movies):")
    print("-" * 40)
    for idx, row in cluster_movies.head(5).iterrows():  # Show first 5 in each cluster
        print(f"  - {row['title']} ({row['genre']})")


### Visualizing Clusters

Let's reduce the dimensions to 2D using PCA (Principal Component Analysis) so we can visualize the clusters:


In [ ]:
# Visualizing Clusters - Concept
# You can try this in Exercise 6 after implementing clustering!

print("=" * 60)
print("Visualizing Clusters with PCA (Concept):")
print("=" * 60)

print("\nProblem: TF-IDF vectors are high-dimensional (100+ dimensions)")
print("  - Can't visualize in 100D space!")
print("  - Need to reduce to 2D for plotting")

print("\nSolution: Principal Component Analysis (PCA)")
print("  - Reduces dimensions while preserving information")
print("  - Projects high-dimensional vectors to 2D")
print("  - Each point in 2D = one document")

print("\nVisualization Process:")
print("  1. Convert sparse TF-IDF matrix to dense")
print("  2. Apply PCA to reduce to 2 dimensions")
print("  3. Plot documents in 2D space")
print("  4. Color by cluster assignment")
print("  5. Add movie titles as labels")

print("\nWhat to look for:")
print("  ✅ Documents in same cluster should be close together")
print("  ✅ Different clusters should be separated")
print("  ✅ Clusters should make semantic sense (similar genres)")

print("\nVariance Explained:")
print("  - PCA preserves as much information as possible")
print("  - Example: 80% variance explained = 80% of information kept")
print("  - Lower variance = more information lost in 2D projection")

print("\n💡 Implementation: After completing Exercise 6, try visualizing clusters!")
print("   Use PCA to reduce dimensions and matplotlib to plot.")


## Summary

In this notebook, you learned:

1. ✅ **TF-IDF**: Improved text representation that weights words by importance
2. ✅ **Similarity-Based Search**: Using TF-IDF + cosine similarity to find relevant documents
3. ✅ **Clustering**: Grouping similar documents with K-Means (unsupervised learning)

### Next Steps

- **Practice**: Complete the Exercise Notebook to implement TF-IDF from scratch!
- **Next Class**: You'll learn about embeddings for true semantic search

**Note**: Production search systems, evaluation metrics, and connections to supervised learning are covered in the theory slides (Class 1.md) - read those for the complete picture!
